In [ ]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

In [ ]:
# Load images → apply transformations → create dataset of all images
class ImageProcessor:
    def __init__(self, root_dir_path, transformations=None):
        self.root_dir_path = root_dir_path         # Store the folder containing images
        self.transformations = transformations     # Store the image transformations

        # Create a list containing the full path of every image
        self.all_img_paths = [
            os.path.join(root_dir_path, img)
            for img in os.listdir(root_dir_path)
        ]

# DataLoader expects 2 imp functions from this class (__len__() & __getitem__())
    # Tells DataLoader how many images are in the dataset
    def __len__(self):
        return len(self.all_img_paths)

    # Tells DataLoader how to get one image
    def __getitem__(self, idx):
        img_path = self.all_img_paths[idx]            # Get the image path at the given index
        img = Image.open(img_path).convert("RGB")     # Open the image and convert it to RGB

        # Apply transformations if they were provided
        if self.transformations:
            img = self.transformations(img)

        # return the processed img
        return img

In [ ]:
root_dir_path = "./img_align_celeba/"

transformations = transforms.Compose([
    transforms.CenterCrop(178),       # by default, celeba dataset images Dimension 178x218. To convert square size we need to crop
    transforms.Resize(64),            # Resize the image from 178×178 → 64×64
    transforms.ToTensor(),            # Convert image from PIL format → PyTorch Tensor
    
    # Normalize pixel values from [0, 1] → [-1, 1]
    transforms.Normalize(
        (0.5, 0.5, 0.5),    # Mean for R, G, B
        (0.5, 0.5, 0.5))    # Std for R, G, B
])

In [ ]:
dataset = ImageProcessor(root_dir_path, transformations)
print(f"Loaded {len(dataset)} images")

In [ ]:
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

#### Generator Network

In [ ]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim=100, img_channels=3):
        super(Generator, self).__init__()

        # fully connected (dense) layers
        self.model = nn.Sequential(
            # Convert random noise: 100 → 256 (upsampling)
            nn.Linear(z_dim, 256),
            nn.ReLU(),                 # Add non-linearity
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 64*64*img_channels),
            nn.Tanh()      # [-1,1]   (to match the real structure)
        )

    # Generate an image from random noise
    def forward(self, z):
        img = self.model(z)       # Pass noise through all layers
        img = img.view(img.size(0), 3, 64, 64)     # Reshape flat values → image: [batch, 3, 64, 64]
        return img

# Fake image => 64 x 64 x 3 x batch_size  (4 Dimensional)

#### Discriminator Network

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, img_channels=3):
        super(Discriminator, self).__init__()

        # fully connected (dense) layers
        self.model = nn.Sequential(
            nn.Flatten(),                      # 4D tensor => 1D
            nn.Linear(64*64*img_channels, 1024),
            nn.LeakyReLU(0.2, inplace=True),   # LeakyReLU prevents neurons from completely "dying" when their input is negative.
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid()      # 0-Fake/1-Real (probability)
        )

    def forward(self, img):
        return self.model(img)

In [ ]:
GAN_loss = nn.BCELoss()     # Binary Cross Entropy loss for Real/Fake classification

generator = Generator()
g_optimizer = optim.Adam(
    generator.parameters(),
    lr=0.0002,              # Learning rate
    betas=(0.5, 0.999)      # Adam momentum settings
)

discriminator = Discriminator()
d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

In [ ]:
# device

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
    
print(f"Device is {device}")

In [ ]:
generator = generator.to(device)
discriminator = discriminator.to(device)

#### Training the GAN

In [ ]:
def train(generator, discriminator, dataloader, epochs=10):

    for epoch in range(epochs):
        for i, imgs in enumerate(dataloader):
            real_imgs = imgs.to(device)
            batch_size = real_imgs.size(0)

            # create real/fake img labels
            real_labels = torch.ones(batch_size, 1).to(device)    # [1,1,1,......]
            fake_labels = torch.zeros(batch_size, 1).to(device)   # [0,0,0,......]

            # train the Discriminator
            d_optimizer.zero_grad()    # reset to zero

            noise = torch.randn(batch_size, 100, device=device)
            fake_imgs = generator(noise)

            real_loss = GAN_loss(discriminator(real_imgs), real_labels)
            fake_loss = GAN_loss(discriminator(fake_imgs.detach()), fake_labels)

            d_loss = (real_loss + fake_loss) / 2

            d_loss.backward()
            d_optimizer.step()

            # train the Generator
            g_optimizer.zero_grad()

            g_loss = GAN_loss(discriminator(fake_imgs), real_labels)

            g_loss.backward()
            g_optimizer.step()

            if i % 50 == 0:
                print(f"for epoch: {epoch+1}/{epochs}... batch: {i+1}... G-loss:{g_loss}.... D-loss: {d_loss}")

        # save generated imgs for each epoch
        save_generated_images(generator, epoch, device)

In [ ]:
import torchvision
import matplotlib.pyplot as plt

def save_generated_images(generator, epoch, device, num_imgs=8):
    z = torch.randn(num_imgs, 100).to(device)
    generated_imgs = generator(z).detach().cpu()

    grid = torchvision.utils.make_grid(generated_imgs, nrow=4, normalize=True)

    plt.imshow(np.transpose(grid, (1, 2, 0)))
    plt.title(f"epoch {epoch+1}")
    plt.axis("off")
    plt.show()

In [ ]:
train(generator, discriminator, dataloader, epochs=5)